# Project 6: Publish a Model: Serving in Python

- Author: Kim Hummel, Denise Case
- Date: 2026-06
- Dataset: Diabetes Data
- Target: diabetic
  
This notebook follows the outline give by Denise Case. It uses a diabetes dataset from Kaggle and attempts to predict if a person is diabetic or not based on their glucose level, BMI, and age. 

Original dataset found here: https://www.kaggle.com/datasets/rishitjakharia/diabetes-prediction?select=diabetes.csv 

## Overview

This project uses a dataset about people's physical characteristics, including whether or not they have diabetes.
We choose to predict the target `diabetic`.
This target is a **discrete category**, so we have a:

- supervised ML problem (because we've chosen a target)
- a classification problem (because our target is a category)


## Section 1. Project Setup and Imports

In [25]:
# === Section 1a. DECLARE IMPORTS ===

from importlib.metadata import version  # to verify
import logging  # for type hinting
import os
from pathlib import Path
import platform  # to verify
from typing import Any  # for type hinting

from datafun_toolkit.logger import get_logger, log_header
import joblib
from model_builder_project06 import (
    DATASET_NAME,
    FEATURE_COLS,
    MODEL_PATH,
    TARGET_COL,
    load_data,
    save_model,
    split_data,
    summarize,
    train_model,
)
import pandas as pd

# Walk up until we find pyproject.toml (project root marker)
while not Path("pyproject.toml").exists():
    os.chdir("..")


from project06.serve_project06 import predict_from_features  # noqa: E402

# === Section 1b. CONFIGURE LOGGER ONCE PER NOTEBOOK ===

LOG: logging.Logger = get_logger("M06", level="DEBUG")
log_header(LOG, "M06")


# === Section 1c. USE THE LOGGER TO VERIFY IMPORTS ===

# If any do NOT return a version number, then that package is not installed correctly.
# Check your pyproject.toml and re-run environment setup commands.

LOG.info("Confirming installation:")
LOG.info(f"  python:       {platform.python_version()}")
LOG.info(f"  pandas:       {version('pandas')}")

# === Section 1d. SET PANDAS DISPLAY CONFIGURATION (helps in notebooks) ===

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

# === Section 1e. GLOBAL CONSTANTS AND CONFIGURATION ===

# CUSTOM: where the published model artifact is written.
LOG.info(f"Model artifact will be saved to: {MODEL_PATH}")

2026-08-09 13:54:19 | INFO | M06 | === RUN START ===
2026-08-09 13:54:19 | INFO | M06 | project=M06
2026-08-09 13:54:19 | INFO | M06 | repo_dir=ml-06-serving
2026-08-09 13:54:19 | INFO | M06 | python=3.14.3
2026-08-09 13:54:19 | INFO | M06 | os=Windows 11
2026-08-09 13:54:19 | INFO | M06 | shell=powershell
2026-08-09 13:54:19 | INFO | M06 | cwd=.
2026-08-09 13:54:19 | INFO | M06 | github_actions=False
2026-08-09 13:54:19 | INFO | M06 | Confirming installation:
2026-08-09 13:54:19 | INFO | M06 |   python:       3.14.3
2026-08-09 13:54:19 | INFO | M06 |   pandas:       3.0.5
2026-08-09 13:54:19 | INFO | M06 | Model artifact will be saved to: artifacts\model2.joblib


## Section 2. Load and Prepare the Model

In [26]:
# === Section 2. Load the Data ===

LOG.info(f"Loading dataset: {DATASET_NAME}")
df_model: pd.DataFrame = load_data()
df_model["diabetic"] = df_model["outcome"].map({0: "Not Diabetic", 1: "Diabetic"})
LOG.info(f"Model rows: {df_model.shape[0]}")
LOG.info(f"Classes in '{TARGET_COL}': {sorted(df_model[TARGET_COL].unique())}")
LOG.info(f"Feature Columns: '{FEATURE_COLS}")

2026-08-09 13:54:26 | INFO | M06 | Loading dataset: diabetes_data
2026-08-09 13:54:26 | INFO | M06 | Loading dataset: diabetes_data
2026-08-09 13:54:26 | INFO | M06 | Loaded: 768 rows, 9 columns
2026-08-09 13:54:26 | INFO | M06 | Model rows (after dropping missing): 768
2026-08-09 13:54:26 | INFO | M06 | Model rows: 768
2026-08-09 13:54:26 | INFO | M06 | Classes in 'diabetic': ['Diabetic', 'Not Diabetic']
2026-08-09 13:54:26 | INFO | M06 | Feature Columns: '['glucose', 'BMI', 'age']


## Section 3. Split into Train and Test

In [27]:
# === Section 3. Split into Train and Test ===

X_train, X_test, y_train, y_test = split_data(df_model)
LOG.info(f"Train instances: {len(X_train)}")
LOG.info(f"Test instances:  {len(X_test)}")

2026-08-09 13:54:30 | INFO | M06 | Train instances: 614
2026-08-09 13:54:30 | INFO | M06 | Test instances:  154
2026-08-09 13:54:30 | INFO | M06 | Train instances: 614
2026-08-09 13:54:30 | INFO | M06 | Test instances:  154


## Section 4. Train, Save, Reload Model 

In [28]:
# === Section 4. Train, Save, and Reload ===

model2 = train_model(X_train, y_train)
save_model(model2)
model2 = joblib.load(MODEL_PATH)
LOG.info(f"Reloaded model from: {MODEL_PATH}")

2026-08-09 13:54:33 | INFO | M06 | Training RandomForestClassifier on 614 instances
2026-08-09 13:54:33 | INFO | M06 | Training complete
2026-08-09 13:54:34 | INFO | M06 | Saved model to: artifacts\model2.joblib
2026-08-09 13:54:34 | INFO | M06 | Reloaded model from: artifacts\model2.joblib


## Section 5. Test the Serving Core

In [29]:
# === Section 5. Test the Serving Core ===

# valid payload - should return a prediction
good_payload: dict[str, Any] = {
    "glucose": 117,
    "BMI": 32,
    "age": 29,
}
result: dict[str, Any] = predict_from_features(model2, good_payload)
LOG.info(f"Valid payload -> {result}")

# invalid payload - should raise a clean ValueError, not crash
bad_payload: dict[str, Any] = {"glucose": 45.0}
try:
    predict_from_features(model2, bad_payload)
    LOG.warning("Expected a ValueError for the bad payload but none was raised.")
except ValueError as exc:
    LOG.info(f"Invalid payload handled cleanly -> ValueError: {exc}")

c:\Repos\Applied_Machine_Learning\ml-06-serving\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
2026-08-09 13:54:37 | INFO | M06 | Valid payload -> {'prediction': 'Not Diabetic'}
2026-08-09 13:54:37 | INFO | M06 | Invalid payload handled cleanly -> ValueError: Missing required feature: 'BMI'


## Section 6. Summary and Next Steps

First, output key information (may use Python)
Second, provide your narrative, conclusions, and next steps (in Markdown)

In [30]:
# === Section 6. Summary ===

# Python summary
summarize()

2026-08-09 13:54:49 | INFO | M06 | ========================
2026-08-09 13:54:49 | INFO | M06 | SUMMARY
2026-08-09 13:54:49 | INFO | M06 | ========================
2026-08-09 13:54:49 | INFO | M06 | Dataset:  diabetes_data
2026-08-09 13:54:49 | INFO | M06 | Target:   diabetic
2026-08-09 13:54:49 | INFO | M06 | Features: ['glucose', 'BMI', 'age']
2026-08-09 13:54:49 | INFO | M06 | Artifact: artifacts\model2.joblib
2026-08-09 13:54:49 | INFO | M06 | ========================



### Custom Narrative & Next Steps 

See the index file to see the custom narrative for phase 5, alongs with next steps to be taken. 

[index.md](docs/index.md)
